# RE_401 · Quins tipus de clients tenim? · resolt

> **Repte resolt.** Una manera de resoldre'l, no l'única. Els números del text surten d'una execució real del quadern.

Treballes per a un distribuïdor d'alimentació de Portugal amb 440 clients (restaurants, hotels,
botigues...). Màrqueting vol fer ofertes diferents per a cada tipus de client, però **no té cap
etiqueta**: només sap quant gasta cada client en sis famílies de productes.

**La pregunta:** quants grups de clients hi ha, i com els descriuries amb paraules? Han de ser grups que
màrqueting pugui fer servir.

L'enunciat complet, amb el diccionari de columnes, és a [RE_401](../RE_401_clients_majorista.ipynb). Si encara no l'has fet, fes-lo abans de llegir això.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

URL = "https://archive.ics.uci.edu/ml/machine-learning-databases/00292/Wholesale%20customers%20data.csv"
df = pd.read_csv(URL)
print(df.shape)
df.head()

In [ ]:
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import adjusted_rand_score, silhouette_score
from sklearn.preprocessing import StandardScaler

COLS = ["Fresh", "Milk", "Grocery", "Frozen", "Detergents_Paper", "Delicassen"]

## 1.1 · La foto

Forma, tipus i nuls. Separa les sis columnes de despesa en una matriu `X`. Per a cada columna,
compara la mitjana, la mediana i el màxim. Què et diu?

In [ ]:
print(df.dtypes.unique(), df.isna().sum().sum())
X = df[COLS]
assert X.shape == (440, 6)
pd.DataFrame({"mitjana": X.mean(), "mediana": X.median(), "max": X.max(),
              "max/mediana": (X.max() / X.median()).round(1)}).round(0)

Totes les columnes tenen la mitjana per sobre de la mediana i un màxim de 13 a 50 vegades la
mediana: distribucions molt **asimètriques**, amb uns pocs clients que gasten moltíssim.

## 1.2 · Com és cada distribució

Dibuixa l'histograma de cada columna, primer tal com ve i després amb `np.log1p`. Quina de les
dues formes et sembla més adequada per mesurar distàncies entre clients?

In [ ]:
fig, eixos = plt.subplots(2, 6, figsize=(18, 5))
for j, c in enumerate(COLS):
    eixos[0, j].hist(X[c], bins=30); eixos[0, j].set_title(c)
    eixos[1, j].hist(np.log1p(X[c]), bins=30)
eixos[0, 0].set_ylabel("tal com ve"); eixos[1, 0].set_ylabel("log1p")
plt.tight_layout()
assert (np.log1p(X) >= 0).all().all()

Tal com venen, tot s'amuntega a l'esquerra amb una cua llarga; amb el logaritme, les formes són
aproximadament simètriques. Per a distàncies, el logaritme fa que «gastar el doble» compti igual
a qualsevol nivell.

## 2.1 · Escalar a mà

Estandarditza `X` **a mà** amb NumPy (restar la mitjana i dividir per la desviació de cada columna)
i compara-ho amb `StandardScaler`. Si no et coincideix, busca per què.

In [ ]:
a_ma = (X - X.mean()) / X.std()
print("amb pandas .std():", np.allclose(a_ma, StandardScaler().fit_transform(X)))
a_ma = (X - X.mean()) / X.std(ddof=0)
assert np.allclose(a_ma, StandardScaler().fit_transform(X))
print("amb ddof=0:", True, "| mitjanes:", a_ma.mean().abs().max().round(12), "| desv:", a_ma.std(ddof=0).round(6).unique())

`X.std()` de pandas divideix per $n-1$ i `StandardScaler` per $n$ (`ddof=0`). Amb 440 files la
diferència és petita però `np.allclose` la detecta.

## 2.2 · Tres versions, set valors de k

Prepara tres versions de les dades: **tal com venen**, **estandarditzades** i **logaritme i després
estandarditzades**. Per a cada una, fes k-means amb k de 2 a 8 i guarda la inèrcia, la silueta i
la mida del grup més petit. Fes una taula i el gràfic del colze i de la silueta.

In [ ]:
versions = {"cru": X.values.astype(float),
            "estandard": StandardScaler().fit_transform(X),
            "log+estandard": StandardScaler().fit_transform(np.log1p(X))}
files, models = [], {}
for nom, Z in versions.items():
    for k in range(2, 9):
        km = KMeans(n_clusters=k, n_init=10, random_state=0).fit(Z)
        models[nom, k] = km
        mides = np.bincount(km.labels_)
        assert mides.sum() == 440
        files.append({"versio": nom, "k": k, "inercia": km.inertia_,
                      "silueta": silhouette_score(Z, km.labels_), "minim": mides.min()})
res = pd.DataFrame(files)
assert len(res) == 21
for nom, g in res.groupby("versio"):
    assert (np.diff(g["inercia"]) <= 1e-6).all()
res.pivot(index="k", columns="versio", values=["silueta", "minim"]).round(3)

In [ ]:
fig, eix = plt.subplots(1, 2, figsize=(11, 4))
for nom, g in res.groupby("versio"):
    eix[0].plot(g["k"], g["inercia"] / g["inercia"].iloc[0], marker="o", label=nom)
    eix[1].plot(g["k"], g["silueta"], marker="o", label=nom)
eix[0].set_title("inèrcia (relativa a k=2)"); eix[1].set_title("silueta"); eix[1].legend()

## 2.3 · La inèrcia, comprovada

Agafa un dels models de la 2.2 i calcula la inèrcia **a mà**, a partir de les etiquetes i dels
centres. Ha de coincidir amb `km.inertia_`.

In [ ]:
Z = versions["log+estandard"]; km = models["log+estandard", 2]
la_meva = ((Z - km.cluster_centers_[km.labels_]) ** 2).sum()
assert np.isclose(la_meva, km.inertia_)
la_meva, km.inertia_

## 2.4 · Quina versió i quina k

Tria una versió i una k. No et quedis només amb la silueta més alta: mira també **quants clients
té cada grup**. Un grup que màrqueting no pot fer servir, és un bon grup?

In [ ]:
for clau in [("cru", 2), ("estandard", 2), ("estandard", 4), ("log+estandard", 2), ("log+estandard", 3)]:
    km = models[clau]
    print(clau, "silueta", round(silhouette_score(versions[clau[0]], km.labels_), 3), "mides", np.bincount(km.labels_))
s = res.set_index(["versio", "k"])["silueta"]
assert s["estandard", 2] > s["log+estandard", 2]

**La trampa:** la silueta més alta és la de les dades sense logaritme (0,51 crues, 0,57
estandarditzades, amb k=2), però els grups són «els que gasten moltíssim» (49 clients) contra
la resta (391), i amb k més gran apareixen grups d'1, 2 o 4 clients: k-means **separa outliers**.
Una silueta alta vol dir grups separats, no grups útils. Amb logaritme la silueta és més baixa
(0,29 a k=2) però els grups són equilibrats. Elecció: **log + estandarditzar, k = 2** (amb k = 3
com a alternativa raonable).

## 2.5 · Posa'ls nom

Descriu cada grup amb **les teves paraules**, a partir de la seva despesa típica en unitats
originals (no estandarditzades). Dona-li un nom a cada un. Només al final, compara els grups amb
`Channel`: s'hi assemblen?

In [ ]:
km = models["log+estandard", 2]
perfil = df.groupby(km.labels_)[COLS].median()
perfil["clients"] = np.bincount(km.labels_)
print(perfil.round(0))
taula = pd.crosstab(km.labels_, df["Channel"])
print(taula)
ari = {nom: adjusted_rand_score(df["Channel"], models[nom, 2].labels_) for nom in versions}
assert ari["log+estandard"] > 0.4 > ari["estandard"]
{k: round(v, 3) for k, v in ari.items()}

Un grup compra sobretot **frescos i congelats** i poc ultramarins, llet i detergents (el perfil
d'un restaurant o un hotel); l'altre compra molts **ultramarins, llet i detergents/paper** (el
perfil d'una botiga). Creuats amb `Channel`, el grup amb logaritme coincideix força amb
hoteleria / detall (ARI ≈ 0,51), mentre que la versió estandarditzada sense logaritme no s'hi
assembla gens (ARI ≈ 0,21). La preparació de les dades ha decidit el resultat més que la k.

## Conclusió

Proposem **dos tipus de clients**, fets amb k-means sobre la despesa amb logaritme i
estandarditzada: un que compra sobretot frescos i congelats (el perfil d'un restaurant o un hotel) i
un altre que compra ultramarins, llet i detergents (el perfil d'una botiga). Sense logaritme la
silueta és més alta (0,57), però els grups són 49 «grans clients» contra 391, i amb més grups en
surten d'1 a 4 clients: k-means separa *outliers*, no tipus de client. Els grups amb logaritme
s'assemblen força a `Channel` (ARI 0,51) i els de sense logaritme, no (0,21). La preparació de les
dades ha pesat més que la tria de k; k = 3 seria una alternativa raonable.